# SHIPP — Run the pipeline end to end (Silver → Gold)

Runs each stage with `dbutils.notebook.run`, stops at the first failed gate, and appends one row per
stage to `pipeline_run_log` (evidence for the Velocity metric).

```text
10 silver_listings → 11 silver_requests → 30 pairs → 31 ORS → 32 routes → 40 gold
```
Set `RUN_ORS = False` to reuse cached routes. **Clear state and outputs → Run all.** Every gate prints PASS or stops. Never loosen an assert.

In [0]:
%run ../common/shipp_silver_lib

In [0]:
import uuid
from datetime import datetime, timezone

RUN_ORS = True
STAGES = [
    ("10_silver_listings", "../development/10_silver_listings_dev"),
    ("11_silver_requests", "../development/11_silver_requests_dev"),
    ("30_candidate_pairs", "../development/30_silver_candidate_pairs"),
    ("31_ors_enrichment",  "../development/31_ors_route_enrichment"),
    ("32_silver_routes",   "../development/32_silver_routes"),
    ("40_gold_matches",    "../development/40_gold_candidate_matches"),
]
if not RUN_ORS:
    STAGES = [s for s in STAGES if s[0] != "31_ors_enrichment"]

LOG_SCHEMA = ("run_id string, stage string, started_at timestamp, ended_at timestamp, "
              "seconds double, status string, summary string")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")
spark.sql(f"CREATE TABLE IF NOT EXISTS {PIPELINE_RUN_LOG} ({LOG_SCHEMA}) USING DELTA")
RUN_ID = str(uuid.uuid4())
print("RUN_ID =", RUN_ID)

In [0]:
log_rows, run_started = [], time.time()
try:
    for stage, path in STAGES:
        started, t0 = datetime.now(timezone.utc).replace(tzinfo=None), time.time()
        print(f"▶ {stage} ...")
        try:
            result, status = dbutils.notebook.run(path, 1800), "PASS"
        except Exception as e:
            result, status = f"{type(e).__name__}: {str(e)[:1000]}", "FAILED"
        seconds = round(time.time() - t0, 1)
        log_rows.append((RUN_ID, stage, started, datetime.now(timezone.utc).replace(tzinfo=None), seconds, status, result))
        print(f"  {status} in {seconds}s — {result}")
        if status == "FAILED":
            raise AssertionError(f"Pipeline stopped at {stage}. Open that notebook and run it to see the failing gate.")
finally:
    if log_rows:
        spark.createDataFrame(log_rows, LOG_SCHEMA).write.format("delta").mode("append").saveAsTable(PIPELINE_RUN_LOG)
    print(f"Total pipeline time: {time.time() - run_started:.1f}s")

print("PASS — pipeline completed.")
display(spark.table(PIPELINE_RUN_LOG).filter(F.col("run_id") == RUN_ID).orderBy("started_at"))